# simws 0.9b/0.9c tour: contacts, small-signal, one geometry, conservative coupling, second engines

Run in the `tcad` kernel from `~/sim_workspace`. Each cell takes seconds unless noted.

In [ ]:
import sys, json; sys.path.insert(0, '..')
import numpy as np
import simws; print(simws.__version__)

## Schottky contact: thermionic emission against its closed form
The deviation of the drift-diffusion device from J = A*T² e^(−φB/kT)(e^(V/kT) − 1) is v_R/v_D ∝ 1/μ, so at μ = 1e8 it is far below the 1e-4 gate.

In [ ]:
from simws.validation import contact_checks as K
r = K.te_closed_form()
print('worst', r['worst_rel_diff'], ' 1/mu ratio', r['deviation_ratio_mu'])

In [ ]:
r = K.tunnel_quadrature(); print('Gamma quadrature vs adaptive:', r['worst_rel_diff'])
r = K.padovani_stratton(); print('ours / Padovani-Stratton in the TFE regime:', r['tfe_regime_ratio_range'])

## Small-signal and noise
Depletion capacitance, then shot noise of an ideal diode (2qI).

In [ ]:
from simws.validation import small_signal_checks as S
print(S.depletion_capacitance()['worst_rel_diff'])
print(S.shot_noise()['worst_rel_diff'])

HBT f_T from AC h21 against charge control (about a minute per point):

In [ ]:
from simws.tcad.hbt import HBTCase
from simws.tcad.small_signal import hbt_small_signal
r = hbt_small_signal(HBTCase(ge_peak=0.15), VBE=0.85, VCE=1.0)
print(f"fT AC {r['fT_ac_Hz']/1e9:.2f} GHz, charge control {r['charge']['fT_charge_Hz']/1e9:.2f} GHz, fmax {r['fmax_Hz']/1e9:.1f} GHz")

## One geometry for every engine
A structured (transfinite) mesh makes DEVSIM's control volumes tile each region exactly.

In [ ]:
from simws.geometry import Geometry
g = Geometry('mos')
g.rect('si', 'Silicon', 0, 1.0, 0, 0.45, size=(0.02, 0.05)).rect('si', 'Silicon', 0, 1.0, 0.45, 0.5, size=(0.02, 0.002))
g.rect('ox', 'Oxide', 0, 1.0, 0.5, 0.52, size=(0.02, 0.002))
g.contact('sub', 0, 1, 0, 0).contact('gate', 0, 1, 0.52, 0.52)
p = g.mesh('/tmp/mos_tour.msh', structured=True)
import simws.tcad
from devsim import get_node_model_values, reset_devsim
reset_devsim(); g.to_devsim(p, device='d')
for r, A in (('si', 0.5), ('ox', 0.02)):
    print(r, sum(get_node_model_values(device='d', region=r, name='NodeVolume')) / (A * 1e-8) - 1)
mesh, sub, bnd = g.to_skfem(p); print('skfem triangles', {k: len(v) for k, v in sub.items()})
print('Geant4 stack at x=0.5:', Geometry('s').rect('a', 'Silicon', 0, 1, 0, 2).geant4_slab(0.5))

## Conservative transfer
A Geant4-style dose grid onto the DEVSIM nodes: the integral is conserved to round-off.

In [ ]:
from simws.validation import geometry_checks as G
r = G.transfer('/tmp/tour_transfer')
print(json.dumps({k: (v['conservation_rel_error'], v['roundtrip_l2_rel']) for k, v in r.items() if isinstance(v, dict)}, indent=1))

## preCICE two-way electro-thermal coupling
DEVSIM and scikit-fem as two processes on different meshes (about 10 s).

In [ ]:
from simws.coupling import precice_bridge as B
from simws.coupling.electrothermal import ResistorCase
r = B.run(ResistorCase(biases=[1.0, 5.0]), thermal_mesh={'size': 0.1})
for p in r['points']:
    print(p['bias_V'], 'V  T_max', round(p['T_max_K'], 4), 'K  power transfer', p['transfer_conservation_rel_error'], ' iterations', p['coupling_iterations'])

## Second engine: DEVSIM vs Solcore
Same 1D diode, same parameters; tolerance (2 %) fixed before the first run.

In [ ]:
from simws.validation import engine_pairs as E
r = E.dd_pair(); print('dark', r['dark_rel'], ' Jsc', r['Jsc']['rel'], ' model-form (relative sigma)', r['model_form_rel_dark'])

## Dask runtime
The same hub jobs, in parallel; identical requests run once and committed jobs are cache hits.

In [ ]:
from simws.hub.runtime import Runtime
with Runtime('/tmp/tour_store', n_workers=2) as rt:
    res = rt.map('materials.intrinsic', [{'material': 'silicon', 'T': (np.array([250., 300.]), 'K')}, {'material': 'silicon', 'T': (np.array([300., 350.]), 'K')}])
print(rt.kind, [r.status for r in res])